## Goal

> ⚠️ **Research on public allocation data.** Not investment advice; results describe the methods, not the funds.

> 🇸🇪 **Categories and cluster names are in Swedish**, as delivered by the source. Common words: *Branschfond* = sector fund, *Sverige* = Sweden, *Norden* = Nordics, *Små-/medelstora bolag* = small/mid caps, *Mix bolag* = mixed, *Råvaror* = commodities, *Läkemedel* = pharma.

Which funds **changed cluster** between two weeks, and was it a real change?

A fund can land in another cluster for two very different reasons:

- **The fund changed**: its country and sector mix moved, so it now looks like other funds.
- **The clustering changed**: the fund barely moved, but the clusters around it split, merged or shifted. That's noise, not news.

This notebook compares one method's results for two weeks you pick, `FROM_WEEK` → `TO_WEEK`, and tells the two apart.

## How the comparison works

1. **Match clusters**: cluster ids are renumbered on every run, so "cluster 12" in one week has nothing to do with "cluster 12" in another. Clusters are paired by the **funds they share** (Jaccard overlap: shared funds ÷ all funds in either). Each cluster is paired at most once, and only when the overlap is at least 0.3. Unpaired clusters **ended** or are **new**, which includes splits and merges.
2. **Per fund**, three numbers:
   - **Mates kept**: overlap of its cluster-mates in the two weeks (0..1). Needs no matching.
   - **Neighbours kept**: share of its 20 most similar funds still in its top 20.
   - **Drift**: how far its own allocation moved, in the method's distance.
3. **Change class** per fund:

| Class | Meaning |
|---|---|
| `stayed` | Still in the matched cluster (or outside a cluster both weeks) |
| `moved` | Other cluster, and drift above the usual: **the holdings changed** |
| `reshuffled` | Other cluster, but normal drift: **the clustering moved, not the fund** |
| `joined` / `left` | Into / out of a cluster ("Not in a cluster" the other week) |
| `new` / `gone` | The fund is in one week's data only |

"Above the usual" = above the 95th percentile of the drift of funds that `stayed`.

In [ ]:
# Re-import edited fund_clustering modules automatically before each cell runs.
%load_ext autoreload
%autoreload 2

In [ ]:
# The two weeks to compare. Each must be run first, one week at a time:
#   python main.py --week 2026-W28   (reads data/, writes output/)
from pathlib import Path

from fund_clustering.load import load_week
from fund_clustering.methods import METHODS
from fund_clustering.migration import load_method_week

DATA_DIR = Path("../data")  # weekly CSV exports (allocations + metadata)
OUTPUT_DIR = Path("../output")  # main.py output for both weeks
FROM_WEEK = "2026-W28"
TO_WEEK = "2026-W45"
METHOD_ID = "hellinger"  # cosine / aitchison / hellinger

method = METHODS[METHOD_ID]
before = load_method_week(OUTPUT_DIR, METHOD_ID, FROM_WEEK)
after = load_method_week(OUTPUT_DIR, METHOD_ID, TO_WEEK)
before_data = load_week(DATA_DIR, FROM_WEEK)
after_data = load_week(DATA_DIR, TO_WEEK)

method.name, len(before.points), len(after.points)

In [ ]:
# Side by side: funds, clusters and funds outside a cluster in each week.
import pandas as pd

from fund_clustering.cluster import NOT_IN_CLUSTER


def week_summary(week) -> dict:
    labels = week.points["cluster"]
    return {
        "funds": len(labels),
        "clusters": labels[labels != NOT_IN_CLUSTER].nunique(),
        "not in a cluster": int((labels == NOT_IN_CLUSTER).sum()),
    }


pd.DataFrame({FROM_WEEK: week_summary(before), TO_WEEK: week_summary(after)})

## Cluster matching

Which cluster in `FROM_WEEK` became which cluster in `TO_WEEK`. A Jaccard of 1 means exactly the same funds. A name change on a matched pair is usually the leading category tipping over, not a new group.

In [ ]:
# Match clusters, classify funds. Drift = how far each fund's own allocation moved.
from fund_clustering.migration import allocation_drift, compare_weeks

drift = allocation_drift(before_data, after_data, method)
migration = compare_weeks(before, after, drift)

before_names = before.points.groupby("cluster")["cluster_name"].first()
after_names = after.points.groupby("cluster")["cluster_name"].first()
before_sizes = before.points["cluster"].value_counts()
after_sizes = after.points["cluster"].value_counts()

matches = migration.matches.assign(
    before_name=lambda m: m["before_cluster"].map(before_names),
    after_name=lambda m: m["after_cluster"].map(after_names),
    before_size=lambda m: m["before_cluster"].map(before_sizes),
    after_size=lambda m: m["after_cluster"].map(after_sizes),
    jaccard=lambda m: m["jaccard"].round(2),
)

# (matched pairs, median overlap, drift threshold for "moved")
print(
    len(matches),
    matches["jaccard"].median(),
    round(migration.drift_threshold, 4),
)
matches.sort_values("jaccard")[
    ["before_name", "before_size", "after_name", "after_size", "jaccard"]
]

In [ ]:
# Clusters without a partner: ended (FROM_WEEK only) and new (TO_WEEK only).
clustered_before = set(before_names.index) - {NOT_IN_CLUSTER}
clustered_after = set(after_names.index) - {NOT_IN_CLUSTER}
ended = sorted(clustered_before - set(migration.matches["before_cluster"]))
new = sorted(clustered_after - set(migration.matches["after_cluster"]))

display(
    pd.DataFrame({"ended": before_names[ended], "size": before_sizes[ended]})
    .sort_values("size", ascending=False)
)
pd.DataFrame({"new": after_names[new], "size": after_sizes[new]}).sort_values(
    "size", ascending=False
)

## Flow

Where each `FROM_WEEK` cluster's funds went. Grey bands stayed, amber were reshuffled by the clustering, red moved because the holdings changed, blue joined or left a cluster. Bands with fewer than `MIN_FLOW` funds are left out to keep it readable.

In [ ]:
# Sankey: FROM_WEEK clusters (left) to TO_WEEK clusters (right).
import plotly.graph_objects as go

MIN_FLOW = 3

CHANGE_COLOURS = {
    "stayed": "rgba(154,160,166,0.45)",
    "reshuffled": "rgba(197,131,0,0.6)",
    "moved": "rgba(208,53,58,0.75)",
    "joined": "rgba(52,120,198,0.55)",
    "left": "rgba(52,120,198,0.55)",
}

both = migration.funds.dropna(subset=["before_cluster", "after_cluster"])
flows = (
    both.groupby(["before_name", "after_name"])
    .agg(funds=("change", "size"), change=("change", lambda c: c.mode()[0]))
    .reset_index()
)
flows = flows[flows["funds"] >= MIN_FLOW]

left = sorted(flows["before_name"].unique())
right = sorted(flows["after_name"].unique())
labels = [f"{name} ({FROM_WEEK})" for name in left] + [
    f"{name} ({TO_WEEK})" for name in right
]
index_left = {name: i for i, name in enumerate(left)}
index_right = {name: len(left) + i for i, name in enumerate(right)}

fig = go.Figure(
    go.Sankey(
        node=dict(label=labels, pad=8, thickness=12, color="#5f6368"),
        link=dict(
            source=flows["before_name"].map(index_left),
            target=flows["after_name"].map(index_right),
            value=flows["funds"],
            color=flows["change"].map(CHANGE_COLOURS),
            customdata=flows["change"],
            hovertemplate="%{value} funds, mostly %{customdata}<extra></extra>",
        ),
    )
)
fig.update_layout(
    title=f"{method.name}: {FROM_WEEK} → {TO_WEEK}", height=max(600, 18 * len(labels))
)
fig.show()

## Fund changes

How many funds fall in each class, then the funds that **moved** (real change), followed by the largest groups that were only reshuffled.

In [ ]:
# Fund table: change class plus display data (name, category) from the CSVs.
display_data = (
    after_data.funds.set_index("isin")[["name", "category"]]
    .combine_first(before_data.funds.set_index("isin")[["name", "category"]])
)
funds = migration.funds.join(display_data)

COLUMNS = [
    "name", "category", "before_name", "after_name",
    "drift", "mate_retention", "neighbour_retention",
]

display(funds["change"].value_counts())

# Moved: other cluster AND the holdings changed more than usual. Largest drift first.
funds.loc[funds["change"] == "moved", COLUMNS].sort_values(
    "drift", ascending=False
).round(3)

In [ ]:
# Reshuffled: other cluster, normal drift. Grouped by from → to cluster, largest first.
(
    funds[funds["change"] == "reshuffled"]
    .groupby(["before_name", "after_name"])
    .agg(
        funds=("name", "size"),
        median_drift=("drift", "median"),
        median_mates_kept=("mate_retention", "median"),
    )
    .sort_values("funds", ascending=False)
    .round(3)
    .head(15)
)

In [ ]:
# Joined / left a cluster: these sit at a cluster's edge, so small shifts flip them.
funds.loc[funds["change"].isin(["joined", "left"]), ["change", *COLUMNS]].sort_values(
    ["change", "drift"], ascending=[True, False]
).round(3)

## Real change or noise?

Each dot is a fund that was in a cluster both weeks. Right = its holdings moved more; down = it lost more of its cluster-mates. The dashed line is the drift threshold. Reshuffled funds sit left of the line yet low on mates kept: the cluster changed around them.

In [ ]:
# Drift vs mates kept, coloured by change class.
import plotly.express as px

scatter = funds.dropna(subset=["mate_retention", "drift"]).reset_index()
fig = px.scatter(
    scatter,
    x="drift",
    y="mate_retention",
    color="change",
    color_discrete_map={
        "stayed": "#9aa0a6", "reshuffled": "#c58300", "moved": "#d0353a"
    },
    hover_name="name",
    hover_data=["isin", "category", "before_name", "after_name", "neighbour_retention"],
    labels={"drift": "Drift (own allocation)", "mate_retention": "Cluster-mates kept"},
    height=600,
)
fig.add_vline(x=migration.drift_threshold, line_dash="dash", line_color="#5f6368")
fig.update_layout(title=f"{method.name}: {FROM_WEEK} → {TO_WEEK}")
fig.show()

## Example funds

The five funds from `explore.ipynb`, from → to cluster and what they kept.

| ISIN | Name |
|------|------|
| `LU0256331488` | Schroder ISF Global Energy A Acc USD |
| `NO0013528349` | DNB European Defence S |
| `LU0069152568` | UBS (Lux) EF Biotech (USD) P-acc |
| `LU0256846568` | Pictet-Security R USD |
| `SE0015837778` | Go Blockchain Fund A |

In [ ]:
# Example funds (see explore.ipynb); try any ISIN here.
EXAMPLE_ISINS = [
    "LU0256331488",  # Schroder ISF Global Energy A Acc USD
    "NO0013528349",  # DNB European Defence S
    "LU0069152568",  # UBS (Lux) EF Biotech (USD) P-acc
    "LU0256846568",  # Pictet-Security R USD
    "SE0015837778",  # Go Blockchain Fund A
]

funds.loc[funds.index.intersection(EXAMPLE_ISINS), ["change", *COLUMNS]].round(3)

## Summary

_Filled in after the first run._